# Train Word LSTM on MediaPipe Keypoints

This notebook:
1. Loads ASL videos from Google Drive
2. Extracts hand keypoints using MediaPipe (robust to background/lighting)
3. Trains an LSTM to classify words based on hand motion

**Why Keypoints?**
- Prevents overfitting to background/signers
- Much faster to train than CNNs
- Focuses purely on hand shape and motion

## 1. Setup

In [ ]:
# Install MediaPipe
!pip install mediapipe

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import random
import numpy as np
import mediapipe as mp
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Extract Videos

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/hf_asl_videos.zip'

if os.path.exists(ZIP_PATH):
    print('Extracting videos...')
    !unzip -q {ZIP_PATH} -d /content/
    print('Done!')
else:
    print(f'ERROR: {ZIP_PATH} not found!')
    print('Please upload hf_asl_videos.zip to your Google Drive root.')

In [ ]:
VIDEO_ROOT = '/content/hf_asl_videos'
words = sorted([d for d in os.listdir(VIDEO_ROOT) if os.path.isdir(os.path.join(VIDEO_ROOT, d))])
print(f'Found {len(words)} words')

## 3. Extract MediaPipe Keypoints

In [ ]:
mp_hands = mp.solutions.hands
mp_pose = mp.solutions.pose

# Config
MAX_FRAMES = 60
KEYPOINTS_ROOT = '/content/keypoints_data'

def extract_landmarks(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    
    if not frames:
        return None

    # Sample frames
    if len(frames) > MAX_FRAMES:
        indices = np.linspace(0, len(frames)-1, MAX_FRAMES, dtype=int)
        frames = [frames[i] for i in indices]
    
    # Initialize MediaPipe
    # We use static_image_mode=False for video tracking (faster & temporal consistency)
    with mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.5) as hands:
        video_landmarks = []
        
        for frame in frames:
            results = hands.process(frame)
            
            # Extract 21 landmarks * 3 coords (x,y,z) * 2 hands = 126 dims
            # Use 0 if hand not detected
            lh = np.zeros(21*3)
            rh = np.zeros(21*3)
            
            if results.multi_hand_landmarks:
                for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                    label = handedness.classification[0].label
                    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]).flatten()
                    
                    if label == 'Left':
                        lh = coords
                    else:
                        rh = coords
            
            # Combine hands (126 dims)
            video_landmarks.append(np.concatenate([lh, rh]))
            
    return np.array(video_landmarks, dtype=np.float32)

# Extract for all videos
all_data = []  # (keypoints, word, original_filename)

for word in tqdm(words, desc='Extracting keypoints'):
    word_dir = os.path.join(VIDEO_ROOT, word)
    videos = [f for f in os.listdir(word_dir) if f.endswith('.mp4')]
    
    for vid in videos:
        kps = extract_landmarks(os.path.join(word_dir, vid))
        if kps is not None and kps.shape[0] > 0:
            all_data.append((kps, word, vid))

print(f'\nExtracted keypoints from {len(all_data)} videos')

## 4. Train/Test Split (Video-level)

In [ ]:
# Split stratified by word
data_by_word = {w: [] for w in words}
for item in all_data:
    data_by_word[item[1]].append(item)

train_data = []
val_data = []
test_data = []

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1

random.seed(42)
for word, items in data_by_word.items():
    random.shuffle(items)
    n = len(items)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    
    train_data.extend(items[:n_train])
    val_data.extend(items[n_train:n_train+n_val])
    test_data.extend(items[n_train+n_val:])

print(f'Train: {len(train_data)}')
print(f'Val:   {len(val_data)}')
print(f'Test:  {len(test_data)}')

## 5. Train LSTM

In [ ]:
# Dataset
class KeypointDataset(Dataset):
    def __init__(self, data, class_to_idx, max_len=60, augment=False):
        self.data = data
        self.class_to_idx = class_to_idx
        self.max_len = max_len
        self.augment = augment
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        kps, word, _ = self.data[idx]
        
        # Data Augmentation
        if self.augment:
            # Random scaling (simulates hand size diffs)
            scale = 1.0 + np.random.uniform(-0.1, 0.1)
            kps = kps * scale
            
            # Random translation (simulates position shifts)
            shift = np.random.uniform(-0.05, 0.05, size=kps.shape[1])
            kps = kps + shift
            
            # Time jitter (drop random frames)
            if len(kps) > 10 and np.random.random() < 0.3:
                drop_idx = np.random.randint(0, len(kps))
                kps = np.delete(kps, drop_idx, axis=0)

        # Pad/Truncate
        if len(kps) > self.max_len:
            kps = kps[:self.max_len]
        elif len(kps) < self.max_len:
            pad = np.zeros((self.max_len - len(kps), 126), dtype=np.float32)
            kps = np.vstack([kps, pad])
            
        label = self.class_to_idx[word]
        return torch.from_numpy(kps).float(), torch.tensor(label, dtype=torch.long)

class_to_idx = {w: i for i, w in enumerate(words)}

train_ds = KeypointDataset(train_data, class_to_idx, augment=True)
val_ds = KeypointDataset(val_data, class_to_idx, augment=False)
test_ds = KeypointDataset(test_data, class_to_idx, augment=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)

In [ ]:
# --- Train Transformer on Keypoints ---
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=60, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class KeypointTransformer(nn.Module):
    def __init__(self, input_dim=126, num_classes=37, d_model=128, nhead=4, num_layers=2, dropout=0.4, max_len=60):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )
        
    def forward(self, x):
        # x: (batch, seq_len, 126)
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.mean(dim=1)  # Global Average Pooling
        return self.classifier(x)

# Initialize Transformer
transformer_model = KeypointTransformer(
    input_dim=126, 
    num_classes=len(words),
    d_model=128,
    nhead=4,
    num_layers=2,
    dropout=0.5
).to(device)

print(f'\nTransformer params: {sum(p.numel() for p in transformer_model.parameters()):,}')

# Train
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(transformer_model.parameters(), lr=5e-4, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

NUM_EPOCHS = 100
best_val_acc = 0
best_transformer_state = None

print('Training Transformer on Keypoints...')
print('=' * 60)

for epoch in range(NUM_EPOCHS):
    transformer_model.train()
    train_correct, train_total = 0, 0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = transformer_model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        train_correct += (out.argmax(1) == y).sum().item()
        train_total += len(y)
    
    scheduler.step()
    
    # Validate
    transformer_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = transformer_model(x)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += len(y)
            
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    
    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_transformer_state = transformer_model.state_dict().copy()
        marker = ' *'
    
    if (epoch + 1) % 10 == 0 or marker:
        print(f'Epoch {epoch+1:3d}: Train={train_acc:.3f}, Val={val_acc:.3f}{marker}')

print('=' * 60)
print(f'Best Val Acc: {best_val_acc:.4f}')

# Test
transformer_model.load_state_dict(best_transformer_state)
transformer_model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        out = transformer_model(x)
        test_correct += (out.argmax(1) == y).sum().item()
        test_total += len(y)

print(f'Test Accuracy: {test_correct/test_total:.4f}')

# Save
torch.save(best_transformer_state, os.path.join(SAVE_DIR, 'best_keypoint_transformer.pth'))
print('Saved Transformer to Google Drive!')

In [ ]:
# LSTM Model
class HandLSTM(nn.Module):
    def __init__(self, input_dim=126, hidden_dim=128, num_classes=37, num_layers=2, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, 
                           batch_first=True, dropout=dropout, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes)
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)
        # Use global max pooling over time (better for alignment issues)
        out = torch.max(out, dim=1)[0]
        return self.fc(out)

model = HandLSTM(num_classes=len(words)).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Train
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=10, factor=0.5, verbose=True)

NUM_EPOCHS = 100
best_val_acc = 0
best_state = None

print('Training LSTM on Keypoints...')
print('=' * 60)

for epoch in range(NUM_EPOCHS):
    model.train()
    train_correct, train_total = 0, 0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        train_correct += (out.argmax(1) == y).sum().item()
        train_total += len(y)
    
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += len(y)
    
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    
    scheduler.step(val_acc)
    
    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = model.state_dict().copy()
        marker = ' *'
        
    if (epoch + 1) % 5 == 0 or marker:
        print(f'Epoch {epoch+1:3d}: Train={train_acc:.3f}, Val={val_acc:.3f}{marker}')

print('=' * 60)
print(f'Best Val Acc: {best_val_acc:.4f}')

In [ ]:
# Test
model.load_state_dict(best_state)
model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        test_correct += (out.argmax(1) == y).sum().item()
        test_total += len(y)

print(f'Test Accuracy: {test_correct/test_total:.4f}')

In [ ]:
# Save
SAVE_DIR = '/content/drive/MyDrive/LearningASL_models'
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(best_state, os.path.join(SAVE_DIR, 'best_keypoint_lstm.pth'))
with open(os.path.join(SAVE_DIR, 'keypoint_classes.txt'), 'w') as f:
    f.write('\n'.join(words))
    
print('Saved model to Google Drive!')